# Water Fixture Classification Pipeline

End-to-end pipeline: label fixture events, fetch magnetometer data, extract features, visualise signal prints, compute priors for the RxInfer generative model.

**Sections:**
1. **Labeling Tool** — record fixture events (start/stop/label)
2. **Fetch & Stats** — pull mag data for each labeled event, extract features
3. **Visualise** — plot signal prints and feature distributions
4. **Extract Priors** — compute prior parameters for the RxInfer model
5. **Model** — the RxInfer fixture classification model (Julia)

In [ ]:
 !pip install requests ipywidgets

In [ ]:
# pip install requests ipywidgets
import requests
import threading
import time
from datetime import datetime, timezone
from IPython.display import display, clear_output
import ipywidgets as widgets

In [ ]:
# ── Hasura connection ─────────────────────────────────────────

HASURA_URL = "https://hasura.pipestuesday.org/v1/graphql"
HASURA_SECRET = "PIPE_SUPERMMMSECRET_PIPE"
SENSOR_ID = 6

def gql(query, variables=None):
    resp = requests.post(
        HASURA_URL,
        json={"query": query, "variables": variables or {}},
        headers={
            "Content-Type": "application/json",
            "x-hasura-admin-secret": HASURA_SECRET,
        },
    )
    data = resp.json()
    if "errors" in data:
        raise Exception(data["errors"])
    return data["data"]

# Test connection
try:
    gql("query { signal(where: {sensor_id: {_eq: 6}}, limit: 1, order_by: {created_at: desc}) { id } }")
    print("Connected to Hasura ✓")
except Exception as e:
    print(f"Failed to connect: {e}")

In [ ]:
# ── GraphQL queries ───────────────────────────────────────────

MUTATION_INSERT = """
mutation InsertSignal(
  $sensor_id: bigint!,
  $start_time: timestamptz!,
  $end_time: timestamptz!,
  $value: String!
) {
  insert_signal_one(object: {
    sensor_id: $sensor_id,
    start_time: $start_time,
    end_time: $end_time,
    value: $value,
    time: $start_time
  }) {
    id
  }
}
"""

QUERY_RECENT = """
query RecentSignals($sensor_id: bigint!) {
  signal(
    where: { sensor_id: { _eq: $sensor_id } }
    order_by: { created_at: desc }
    limit: 5
  ) {
    id
    value
    start_time
    end_time
  }
}
"""

MUTATION_DELETE = """
mutation DeleteSignal($id: bigint!) {
  delete_signal_by_pk(id: $id) {
    id
  }
}
"""

def fetch_recent():
    try:
        return gql(QUERY_RECENT, {"sensor_id": SENSOR_ID})["signal"]
    except Exception:
        return []

def save_signal(start_time, end_time, label):
    st = start_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    et = end_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    data = gql(MUTATION_INSERT, {
        "sensor_id": SENSOR_ID,
        "start_time": st,
        "end_time": et,
        "value": label,
    })
    return data["insert_signal_one"]["id"]

def delete_signal(signal_id):
    gql(MUTATION_DELETE, {"id": signal_id})

In [ ]:
# ── Labeling UI ───────────────────────────────────────────────

# Formatting helpers
def fmt_time(iso_str):
    if isinstance(iso_str, datetime):
        dt = iso_str.astimezone()
    else:
        dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00")).astimezone()
    return dt.strftime("%H:%M:%S")

def fmt_duration(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    m, s = divmod(seconds, 60)
    return f"{m}m{s:02d}s"

# State
start_time = None
end_time = None
timer_running = False

# ── Styles ─────────────────────────────────────────────────────
HEADER_STYLE = "font-size:18px; font-weight:bold; font-family:monospace; padding:8px 0;"
STATUS_STYLE = "font-size:15px; font-family:monospace; padding:4px 0;"
TABLE_STYLE = "font-family:monospace; font-size:13px;"
BTN_LAYOUT = widgets.Layout(width="120px", height="44px")
BTN_LAYOUT_WIDE = widgets.Layout(width="180px", height="44px")

# ── Widgets ─────────────────────────────────────────────────────
header = widgets.HTML(
    f'<div style="{HEADER_STYLE}">'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>'
    '&nbsp;Water Fixture Labeler &nbsp;|&nbsp; sensor 6<br>'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</div>'
)

status_html = widgets.HTML(value="")
recent_html = widgets.HTML(value="")
message_html = widgets.HTML(value="")

btn_start = widgets.Button(description="⏺ Start", button_style="danger", layout=BTN_LAYOUT_WIDE)
btn_stop = widgets.Button(description="⏹ Stop", button_style="warning", layout=BTN_LAYOUT_WIDE)

btn_toilet = widgets.Button(description="🚽 Toilet", button_style="info", layout=BTN_LAYOUT)
btn_sink = widgets.Button(description="🚰 Sink", button_style="info", layout=BTN_LAYOUT)
btn_shower = widgets.Button(description="🚿 Shower", button_style="info", layout=BTN_LAYOUT)
btn_urinal = widgets.Button(description="🚾 Urinal", button_style="info", layout=BTN_LAYOUT)
btn_discard = widgets.Button(description="✕ Discard", button_style="", layout=BTN_LAYOUT)

btn_undo = widgets.Button(description="↩ Undo last", button_style="warning", layout=BTN_LAYOUT_WIDE)

label_buttons = widgets.HBox([btn_toilet, btn_sink, btn_shower, btn_urinal, btn_discard])
idle_buttons = widgets.HBox([btn_start, btn_undo])
recording_buttons = widgets.HBox([btn_stop])

# All panels stacked
ui = widgets.VBox([
    header,
    message_html,
    status_html,
    idle_buttons,
    recording_buttons,
    label_buttons,
    recent_html,
])

# ── Render helpers ──────────────────────────────────────────────

def render_recent():
    rows = fetch_recent()
    if not rows:
        recent_html.value = f'<div style="{TABLE_STYLE}; padding-top:12px;">Recent labels: (none)</div>'
        return rows
    lines = ['<div style="padding-top:12px;">', f'<table style="{TABLE_STYLE}; border-collapse:collapse;">']
    lines.append('<tr style="border-bottom:1px solid #555;"><th style="padding:2px 8px; text-align:left;">ID</th>'
                 '<th style="padding:2px 8px; text-align:left;">Label</th>'
                 '<th style="padding:2px 8px; text-align:left;">Time</th>'
                 '<th style="padding:2px 8px; text-align:left;">Duration</th></tr>')
    for sig in rows:
        if sig["start_time"] is None or sig["end_time"] is None:
            st_str = fmt_time(sig["start_time"]) if sig["start_time"] else "—"
            et_str = fmt_time(sig["end_time"]) if sig["end_time"] else "—"
            dur = "—"
        else:
            st = datetime.fromisoformat(sig["start_time"].replace("Z", "+00:00"))
            et = datetime.fromisoformat(sig["end_time"].replace("Z", "+00:00"))
            dur = fmt_duration((et - st).total_seconds())
            st_str = fmt_time(sig["start_time"])
            et_str = fmt_time(sig["end_time"])
        lines.append(
            f'<tr><td style="padding:2px 8px;">#{sig["id"]}</td>'
            f'<td style="padding:2px 8px;">{sig["value"]}</td>'
            f'<td style="padding:2px 8px;">{st_str} → {et_str}</td>'
            f'<td style="padding:2px 8px;">{dur}</td></tr>'
        )
    lines.append('</table></div>')
    recent_html.value = '\n'.join(lines)
    return rows

def show_state_idle(msg=None):
    global start_time, end_time
    start_time = None
    end_time = None
    message_html.value = f'<div style="{STATUS_STYLE}; color:#4a4;">{msg}</div>' if msg else ""
    status_html.value = f'<div style="{STATUS_STYLE}">Press <b>Start</b> to begin recording a fixture event.</div>'
    idle_buttons.layout.display = "flex"
    recording_buttons.layout.display = "none"
    label_buttons.layout.display = "none"
    render_recent()

def show_state_recording():
    message_html.value = ""
    idle_buttons.layout.display = "none"
    recording_buttons.layout.display = "flex"
    label_buttons.layout.display = "none"
    recent_html.value = ""

def show_state_labeling():
    dur = fmt_duration((end_time - start_time).total_seconds())
    status_html.value = (
        f'<div style="{STATUS_STYLE}">'
        f'Recorded: <b>{fmt_time(start_time)} → {fmt_time(end_time)}</b> &nbsp; ({dur})<br><br>'
        f'What was it?</div>'
    )
    message_html.value = ""
    idle_buttons.layout.display = "none"
    recording_buttons.layout.display = "none"
    label_buttons.layout.display = "flex"
    recent_html.value = ""

# ── Timer ───────────────────────────────────────────────────────

def run_timer():
    global timer_running
    while timer_running:
        elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()
        m, s = divmod(int(elapsed), 60)
        status_html.value = (
            f'<div style="{STATUS_STYLE}">'
            f'<span style="color:#e44;">⏺ RECORDING</span> &nbsp; <b>{m:02d}:{s:02d}</b><br><br>'
            f'Started: {fmt_time(start_time)}<br>'
            f'Go use the fixture, then come back and press <b>Stop</b>.</div>'
        )
        time.sleep(1)

# ── Button callbacks ───────────────────────────────────────────

def on_start(_):
    global start_time, timer_running
    start_time = datetime.now(timezone.utc)
    timer_running = True
    show_state_recording()
    threading.Thread(target=run_timer, daemon=True).start()

def on_stop(_):
    global end_time, timer_running
    timer_running = False
    end_time = datetime.now(timezone.utc)
    show_state_labeling()

def make_label_callback(label):
    def on_label(_):
        try:
            rid = save_signal(start_time, end_time, label)
            show_state_idle(f"✓ Saved as {label.upper()} (id: {rid})")
        except Exception as e:
            message_html.value = f'<div style="{STATUS_STYLE}; color:#e44;">✗ Save failed: {e}</div>'
    return on_label

def on_discard(_):
    show_state_idle("Discarded.")

def on_undo(_):
    rows = fetch_recent()
    if not rows:
        message_html.value = f'<div style="{STATUS_STYLE}">Nothing to undo.</div>'
        return
    last = rows[0]
    try:
        delete_signal(last["id"])
        show_state_idle(f"✓ Deleted #{last['id']}")
    except Exception as e:
        message_html.value = f'<div style="{STATUS_STYLE}; color:#e44;">✗ Undo failed: {e}</div>'

btn_start.on_click(on_start)
btn_stop.on_click(on_stop)
btn_toilet.on_click(make_label_callback("toilet"))
btn_sink.on_click(make_label_callback("sink"))
btn_shower.on_click(make_label_callback("shower"))
btn_urinal.on_click(make_label_callback("urinal"))
btn_discard.on_click(on_discard)
btn_undo.on_click(on_undo)

# ── Launch ─────────────────────────────────────────────────────
show_state_idle()
display(ui)

---

# 2. Fetch & Stats

Fetch all labeled signal windows and the magnetometer readings inside each window. Extract features from each event. Print a statistical summary.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import HTML

# ── Queries ───────────────────────────────────────────────────

QUERY_SIGNALS = """
query GetSignals {
  signal(
    where: {
      sensor_id: { _eq: 6 }
      value: { _is_null: false }
      start_time: { _is_null: false }
      end_time: { _is_null: false }
    }
    order_by: { start_time: asc }
  ) {
    id
    value
    start_time
    end_time
  }
}
"""

QUERY_MAG_REPORTS = """
query GetMagReports($sensor_id: bigint!, $start: timestamptz!, $end: timestamptz!) {
  mag_report(
    where: {
      sensor_id: { _eq: $sensor_id }
      created_at: { _gte: $start, _lte: $end }
    }
    order_by: { created_at: asc }
  ) {
    id
    created_at
    dominant_freq_hz
    flow_running
    flow_intensity
    total_magnitude
    vibration_rpm
    band_energy_10s
    x_axis_reading
    y_axis_reading
    z_axis_reading
  }
}
"""

# ── Fetch labeled signals ─────────────────────────────────────

print("Fetching labeled signals...")
signals = gql(QUERY_SIGNALS)["signal"]
print(f"Found {len(signals)} labeled signal(s)\n")

if not signals:
    raise SystemExit("No labeled signals found. Use the labeling tool above first.")

events = []
event_readings = {}
event_dfs = {}

for sig in signals:
    sid = sig["id"]
    print(f"  Fetching mag_reports for signal #{sid} ({sig['value']})...")
    readings = gql(QUERY_MAG_REPORTS, {
        "sensor_id": SENSOR_ID,
        "start": sig["start_time"],
        "end": sig["end_time"],
    })["mag_report"]

    if len(readings) < 5:
        print(f"  ⚠ Signal #{sid}: only {len(readings)} readings, skipping (need ≥5)")
        continue

    event_readings[sid] = readings
    events.append(sig)

print(f"\n{len(events)} event(s) with sufficient data")

In [ ]:
# ── Feature extraction ────────────────────────────────────────

# Vibration intensity threshold (from pipe project flowComputation.ts)
# Below this, dominant_freq_hz is measuring noise, not real cog rotation.
MIN_VIBRATION = 5.0

def get_vibration(row):
    """Get vibration intensity, preferring 10s window, falling back to 60s."""
    v = row.get("band_energy_10s")
    if v is not None:
        return float(v)
    v = row.get("band_energy_60s")
    if v is not None:
        return float(v)
    return 0.0

def extract_features(sig, readings):
    df = pd.DataFrame(readings)
    df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601")
    t0 = df["created_at"].iloc[0]
    df["elapsed_seconds"] = (df["created_at"] - t0).dt.total_seconds()

    # Vibration intensity per reading
    df["vibration_intensity"] = df.apply(get_vibration, axis=1)
    df["has_real_signal"] = df["vibration_intensity"] >= MIN_VIBRATION

    duration_seconds = df["elapsed_seconds"].iloc[-1]
    total_readings = len(df)

    freq = df["dominant_freq_hz"].values.astype(float)
    elapsed = df["elapsed_seconds"].values
    time_deltas = np.diff(elapsed)

    # ── Vibration-gated frequency ─────────────────────────────
    # Only trust dominant_freq_hz when vibration is above threshold.
    # When below threshold, treat frequency as 0 (noise).
    gated_freq = np.where(df["has_real_signal"].values, freq, 0.0)

    # Estimated cycles: integrate GATED freq over time
    gated_pairs = (gated_freq[:-1] + gated_freq[1:]) / 2.0
    estimated_cycles = float(np.sum(gated_pairs * time_deltas))

    # Middle 60% window for plateau stats (using gated freq)
    t_start_plateau = duration_seconds * 0.2
    t_end_plateau = duration_seconds * 0.8
    plateau_mask = (df["elapsed_seconds"] >= t_start_plateau) & (df["elapsed_seconds"] <= t_end_plateau)
    plateau_signal_mask = plateau_mask & df["has_real_signal"]
    plateau_freq = df.loc[plateau_signal_mask, "dominant_freq_hz"].astype(float)

    if len(plateau_freq) > 0:
        mean_plateau_freq_hz = float(plateau_freq.mean())
        std_plateau_freq_hz = float(plateau_freq.std(ddof=0)) if len(plateau_freq) > 1 else 0.0
    else:
        # Fall back to all readings with real signal
        real_freq = df.loc[df["has_real_signal"], "dominant_freq_hz"].astype(float)
        if len(real_freq) > 0:
            mean_plateau_freq_hz = float(real_freq.mean())
            std_plateau_freq_hz = float(real_freq.std(ddof=0)) if len(real_freq) > 1 else 0.0
        else:
            mean_plateau_freq_hz = float(freq.mean())
            std_plateau_freq_hz = 0.0

    # Peak freq (only from readings with real signal)
    real_signal_freq = freq[df["has_real_signal"].values]
    peak_freq_hz = float(real_signal_freq.max()) if len(real_signal_freq) > 0 else float(freq.max())

    # Ramp duration
    threshold_80 = 0.8 * peak_freq_hz
    ramp_suspicious = False
    reached_mask = (df["dominant_freq_hz"].astype(float) >= threshold_80) & df["has_real_signal"]
    if reached_mask.any():
        first_reach_idx = reached_mask.idxmax()
        ramp_duration_seconds = float(df.loc[first_reach_idx, "elapsed_seconds"])
    else:
        ramp_duration_seconds = duration_seconds
        ramp_suspicious = True

    # Cutoff duration
    if reached_mask.any():
        last_above = reached_mask[::-1].idxmax()
        cutoff_duration_seconds = duration_seconds - float(df.loc[last_above, "elapsed_seconds"])
    else:
        cutoff_duration_seconds = 0.0

    # Freq stability (gated)
    freq_stability = (std_plateau_freq_hz / mean_plateau_freq_hz) if mean_plateau_freq_hz > 0 else None

    # Flow stats
    flow_intensity_vals = df["flow_intensity"].astype(float)
    mean_flow_intensity = float(flow_intensity_vals.mean())
    peak_flow_intensity = float(flow_intensity_vals.max())
    mean_magnitude = float(df["total_magnitude"].astype(float).mean())
    flow_running_vals = df["flow_running"].astype(bool)
    flow_running_fraction = float(flow_running_vals.sum()) / total_readings

    # ── Vibration intensity features ──────────────────────────
    vib = df["vibration_intensity"].values
    mean_vibration_intensity = float(vib.mean())
    peak_vibration_intensity = float(vib.max())

    # Signal fraction: what % of readings have vibration above threshold
    signal_fraction = float(df["has_real_signal"].sum()) / total_readings

    # Signal quality: mean vibration during real-signal readings only
    real_vib = vib[df["has_real_signal"].values]
    mean_signal_vibration = float(real_vib.mean()) if len(real_vib) > 0 else 0.0

    # Plateau vibration: mean vibration in the middle 60% with real signal
    plateau_vib = df.loc[plateau_signal_mask, "vibration_intensity"].values
    mean_plateau_vibration = float(plateau_vib.mean()) if len(plateau_vib) > 0 else 0.0

    # ── Waveform shape features (from raw x/y/z) ─────────
    x_raw = df["x_axis_reading"].astype(float).values
    y_raw = df["y_axis_reading"].astype(float).values
    z_raw = df["z_axis_reading"].astype(float).values
    x_ac = x_raw - x_raw.mean()
    y_ac = y_raw - y_raw.mean()
    z_ac = z_raw - z_raw.mean()

    # Autocorrelation lag 1 — low = fast oscillation (toilet), high = smooth (sink)
    ac1_x = float(np.corrcoef(x_ac[:-1], x_ac[1:])[0, 1]) if len(x_ac) > 1 else 0.0
    ac1_y = float(np.corrcoef(y_ac[:-1], y_ac[1:])[0, 1]) if len(y_ac) > 1 else 0.0
    ac1_z = float(np.corrcoef(z_ac[:-1], z_ac[1:])[0, 1]) if len(z_ac) > 1 else 0.0
    mean_autocorr_lag1 = (ac1_x + ac1_y + ac1_z) / 3.0

    # Autocorrelation lag 5 — captures medium-range oscillation structure
    ac5_x = float(np.corrcoef(x_ac[:-5], x_ac[5:])[0, 1]) if len(x_ac) > 5 else 0.0
    ac5_y = float(np.corrcoef(y_ac[:-5], y_ac[5:])[0, 1]) if len(y_ac) > 5 else 0.0
    ac5_z = float(np.corrcoef(z_ac[:-5], z_ac[5:])[0, 1]) if len(z_ac) > 5 else 0.0
    mean_autocorr_lag5 = (ac5_x + ac5_y + ac5_z) / 3.0

    # Zero-crossing rate — proxy for oscillation frequency
    zcr_x = float(np.sum(np.diff(np.sign(x_ac)) != 0)) / len(x_ac) if len(x_ac) > 1 else 0.0
    zcr_y = float(np.sum(np.diff(np.sign(y_ac)) != 0)) / len(y_ac) if len(y_ac) > 1 else 0.0
    zcr_z = float(np.sum(np.diff(np.sign(z_ac)) != 0)) / len(z_ac) if len(z_ac) > 1 else 0.0
    mean_zero_crossing_rate = (zcr_x + zcr_y + zcr_z) / 3.0

    # Mean peak-to-peak amplitude in 1-second windows
    win = 20  # ~1 second at 50ms intervals
    p2p_vals = []
    for ax in [x_ac, y_ac, z_ac]:
        for i in range(0, len(ax) - win, win):
            p2p_vals.append(ax[i:i+win].max() - ax[i:i+win].min())
    mean_peak_to_peak = float(np.mean(p2p_vals)) if p2p_vals else 0.0

    # Handle NaN from constant signals
    if np.isnan(mean_autocorr_lag1): mean_autocorr_lag1 = 0.0
    if np.isnan(mean_autocorr_lag5): mean_autocorr_lag5 = 0.0
    if np.isnan(mean_zero_crossing_rate): mean_zero_crossing_rate = 0.0

    features = {
        "signal_id": sig["id"],
        "fixture_type": sig["value"],
        "start_time": sig["start_time"],
        "end_time": sig["end_time"],
        "duration_seconds": round(duration_seconds, 2),
        "total_readings": total_readings,
        "estimated_cycles": round(estimated_cycles, 2),
        "mean_plateau_freq_hz": round(mean_plateau_freq_hz, 4),
        "std_plateau_freq_hz": round(std_plateau_freq_hz, 4),
        "peak_freq_hz": round(peak_freq_hz, 4),
        "ramp_duration_seconds": round(ramp_duration_seconds, 2),
        "cutoff_duration_seconds": round(cutoff_duration_seconds, 2),
        "freq_stability": round(freq_stability, 4) if freq_stability is not None else None,
        "mean_flow_intensity": round(mean_flow_intensity, 4),
        "peak_flow_intensity": round(peak_flow_intensity, 4),
        "mean_magnitude": round(mean_magnitude, 2),
        "flow_running_fraction": round(flow_running_fraction, 4),
        # Vibration intensity features
        "mean_vibration_intensity": round(mean_vibration_intensity, 4),
        "peak_vibration_intensity": round(peak_vibration_intensity, 4),
        "signal_fraction": round(signal_fraction, 4),
        "mean_signal_vibration": round(mean_signal_vibration, 4),
        "mean_plateau_vibration": round(mean_plateau_vibration, 4),
        # Waveform shape features
        "mean_autocorr_lag1": round(mean_autocorr_lag1, 4),
        "mean_autocorr_lag5": round(mean_autocorr_lag5, 4),
        "mean_zero_crossing_rate": round(mean_zero_crossing_rate, 4),
        "mean_peak_to_peak": round(mean_peak_to_peak, 2),
    }
    return features, df, ramp_suspicious


# ── Process all events ────────────────────────────────────────

feature_rows = []
warn_list = []
os.makedirs("events", exist_ok=True)

for sig in events:
    sid = sig["id"]
    readings = event_readings[sid]
    features, df, ramp_suspicious = extract_features(sig, readings)
    feature_rows.append(features)
    event_dfs[sid] = df

    # Save per-event CSV (now includes vibration columns)
    cols = ["elapsed_seconds", "dominant_freq_hz", "flow_intensity",
            "total_magnitude", "flow_running", "vibration_rpm",
            "x_axis_reading", "y_axis_reading", "z_axis_reading",
            "vibration_intensity", "has_real_signal"]
    df[cols].to_csv(f"events/signal_{sid}_{sig['value']}.csv", index=False)

    # Warnings
    if features["duration_seconds"] < 3:
        warn_list.append(f"Signal #{sid}: duration {features['duration_seconds']}s < 3s")
    if features["flow_running_fraction"] < 0.3:
        warn_list.append(f"Signal #{sid}: flow_running_fraction {features['flow_running_fraction']:.2f} < 0.3")
    if features["estimated_cycles"] < 1:
        warn_list.append(f"Signal #{sid}: estimated_cycles {features['estimated_cycles']:.1f} < 1")
    if ramp_suspicious:
        warn_list.append(f"Signal #{sid}: ramp never reached 80% of peak freq")
    if features["signal_fraction"] < 0.3:
        warn_list.append(f"Signal #{sid}: signal_fraction {features['signal_fraction']:.2f} — most readings below vibration threshold (noise)")
    if features["mean_vibration_intensity"] < MIN_VIBRATION:
        warn_list.append(f"Signal #{sid}: mean vibration {features['mean_vibration_intensity']:.1f} < {MIN_VIBRATION} — may be entirely noise")

events_df = pd.DataFrame(feature_rows)
events_df.to_csv("events.csv", index=False)

print(f"Saved events.csv ({len(feature_rows)} rows)")
print(f"Saved events/ ({len(feature_rows)} per-event CSVs)")

In [ ]:
# ── Stats summary ─────────────────────────────────────────────

FEATURE_COLS = [
    "duration_seconds", "total_readings", "estimated_cycles",
    "mean_plateau_freq_hz", "std_plateau_freq_hz", "peak_freq_hz",
    "ramp_duration_seconds", "cutoff_duration_seconds", "freq_stability",
    "mean_flow_intensity", "peak_flow_intensity", "mean_magnitude",
    "flow_running_fraction",
    # Vibration intensity features
    "mean_vibration_intensity", "peak_vibration_intensity",
    "signal_fraction", "mean_signal_vibration", "mean_plateau_vibration",
    # Waveform shape features
    "mean_autocorr_lag1", "mean_autocorr_lag5",
    "mean_zero_crossing_rate", "mean_peak_to_peak",
]

type_counts = events_df["fixture_type"].value_counts()

# Summary
html_parts = ['<div style="font-family:monospace; font-size:14px;">']
html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br>')
html_parts.append('<b>&nbsp;Fixture Event Summary</b><br>')
html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br><br>')
for ftype in sorted(type_counts.index):
    n = type_counts[ftype]
    label = "event" if n == 1 else "events"
    html_parts.append(f'&nbsp;&nbsp;{ftype:<12s} {n} {label}<br>')
html_parts.append('<br>')

# Feature stats per type
for ftype in sorted(type_counts.index):
    subset = events_df[events_df["fixture_type"] == ftype]
    n = len(subset)
    html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br>')
    html_parts.append(f'<b>&nbsp;Feature Statistics — {ftype} &nbsp;(n={n})</b><br>')
    html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br><br>')
    html_parts.append('<table style="font-family:monospace; font-size:13px;">')
    for col in FEATURE_COLS:
        vals = subset[col].dropna()
        if len(vals) == 0:
            val_str = "—"
        elif n == 1:
            val_str = f"{vals.iloc[0]}&nbsp;&nbsp;<i>(n=1, more data needed)</i>"
        else:
            m = vals.mean()
            s = vals.std(ddof=1)
            val_str = f"{m:.4f} ± {s:.4f}"
        html_parts.append(f'<tr><td style="padding:1px 8px; color:#6af;">{col}</td>'
                         f'<td style="padding:1px 8px;">{val_str}</td></tr>')
    html_parts.append('</table><br>')

# Warnings
html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br>')
html_parts.append('<b>&nbsp;Warnings</b><br>')
html_parts.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br><br>')
if warn_list:
    for w in warn_list:
        html_parts.append(f'&nbsp;&nbsp;<span style="color:#ea0;">⚠ {w}</span><br>')
else:
    html_parts.append('&nbsp;&nbsp;None<br>')
html_parts.append('</div>')

display(HTML('\n'.join(html_parts)))

---

# 3. Visualise

Plot the signal "prints" (dominant_freq_hz over time) for each fixture type, feature distributions, separation scatter, and raw magnetometer waveforms.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})

os.makedirs("plots", exist_ok=True)

FIXTURE_COLORS = {
    "toilet": "#e74c3c",
    "sink": "#3498db",
    "shower": "#2ecc71",
    "urinal": "#f39c12",
}

fixture_types = sorted(events_df["fixture_type"].unique())

def get_color(ftype):
    return FIXTURE_COLORS.get(ftype, "#999999")

In [ ]:
# ── Plot 1: Signal prints with vibration gating ──────────────

n_types = max(len(fixture_types), 1)
fig, axes = plt.subplots(2, n_types, figsize=(6 * n_types, 7), squeeze=False,
                         gridspec_kw={"height_ratios": [2, 1]})

for idx, ftype in enumerate(fixture_types):
    ax_freq = axes[0, idx]
    ax_vib = axes[1, idx]
    subset = events_df[events_df["fixture_type"] == ftype]
    n = len(subset)
    color = get_color(ftype)

    for _, row in subset.iterrows():
        sid = row["signal_id"]
        df = event_dfs[sid]
        elapsed = df["elapsed_seconds"]
        freq_vals = df["dominant_freq_hz"].astype(float)
        vib_vals = df["vibration_intensity"]
        real = df["has_real_signal"]

        # Top: frequency — solid where real signal, faded where noise
        ax_freq.plot(elapsed[real], freq_vals[real],
                     color=color, alpha=0.6, linewidth=1.2)
        ax_freq.plot(elapsed[~real], freq_vals[~real],
                     color="gray", alpha=0.2, linewidth=0.8, linestyle=":")

        # Bottom: vibration intensity
        ax_vib.plot(elapsed, vib_vals, color=color, alpha=0.5, linewidth=1)

    # Plateau line
    mean_plat = subset["mean_plateau_freq_hz"].mean()
    ax_freq.axhline(mean_plat, color=color, linestyle="--", alpha=0.7,
                    label=f"plateau={mean_plat:.3f}")
    ax_freq.set_title(f"{ftype} (n={n})")
    ax_freq.set_ylabel("dominant_freq_hz")
    ax_freq.legend(fontsize=8)

    # Vibration threshold line
    ax_vib.axhline(MIN_VIBRATION, color="red", linestyle="--", alpha=0.5,
                   label=f"threshold={MIN_VIBRATION}")
    ax_vib.set_ylabel("vibration intensity")
    ax_vib.set_xlabel("elapsed seconds")
    ax_vib.legend(fontsize=8)

fig.suptitle("Signal Prints — frequency (top) & vibration intensity (bottom)\n"
             "Gray dotted = below vibration threshold (noise)", fontweight="bold")
fig.tight_layout()
fig.savefig("plots/raw_signals.png")
print("Saved plots/raw_signals.png")
plt.show()

In [ ]:
# ── Plot 2: Feature distributions ─────────────────────────────

dist_features = ["duration_seconds", "mean_plateau_freq_hz", "estimated_cycles",
                  "ramp_duration_seconds", "peak_freq_hz",
                  "mean_vibration_intensity", "signal_fraction"]

n_feat = len(dist_features)
fig, axes = plt.subplots(n_types, n_feat, figsize=(3 * n_feat, 3 * n_types), squeeze=False)

for row_idx, ftype in enumerate(fixture_types):
    subset = events_df[events_df["fixture_type"] == ftype]
    n = len(subset)
    color = get_color(ftype)

    for col_idx, feat in enumerate(dist_features):
        ax = axes[row_idx, col_idx]
        vals = subset[feat].dropna().values

        if n == 1:
            ax.axvline(vals[0], color=color, linewidth=2)
            ax.set_title(f"{ftype}\n{feat}\nn=1", fontsize=8)
        else:
            ax.hist(vals, bins=max(5, n // 2), color=color, alpha=0.7, edgecolor="white")
            ax.set_title(f"{ftype}\n{feat}", fontsize=8)

        if row_idx == n_types - 1:
            ax.set_xlabel(feat, fontsize=7)

fig.suptitle("Feature Distributions by Fixture Type", fontweight="bold")
fig.tight_layout()
fig.savefig("plots/feature_distributions.png")
print("Saved plots/feature_distributions.png")
plt.show()

In [ ]:
# ── Plot 3: Separation scatter ────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
for ftype in fixture_types:
    subset = events_df[events_df["fixture_type"] == ftype]
    ax.scatter(subset["duration_seconds"], subset["mean_plateau_freq_hz"],
               c=get_color(ftype), label=ftype, s=60, edgecolors="white", zorder=3)

ax.set_xlabel("duration_seconds")
ax.set_ylabel("mean_plateau_freq_hz")
ax.set_title("Primary separation dimensions", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("plots/separation.png")
print("Saved plots/separation.png")
plt.show()

In [ ]:
# ── Plot 4: Raw magnetometer grid ─────────────────────────────

raw_cols = ["x_axis_reading", "y_axis_reading", "z_axis_reading", "total_magnitude"]
# Pick up to 6 most recent events
recent_events = events[-6:]
n_events_to_plot = len(recent_events)

fig, axes = plt.subplots(4, n_events_to_plot, figsize=(4 * n_events_to_plot, 10), squeeze=False)

for col_idx, sig in enumerate(recent_events):
    sid = sig["id"]
    df = event_dfs[sid]
    color = get_color(sig["value"])

    for row_idx, col_name in enumerate(raw_cols):
        ax = axes[row_idx, col_idx]
        vals = df[col_name].astype(float)
        ax.plot(df["elapsed_seconds"], vals, color=color, linewidth=0.6)
        if col_idx == 0:
            ax.set_ylabel(col_name, fontsize=8)
        if row_idx == 0:
            ax.set_title(f"#{sid} {sig['value']}", fontsize=9)
        if row_idx == 3:
            ax.set_xlabel("elapsed_seconds", fontsize=8)

fig.suptitle("Raw Magnetometer Waveforms", fontweight="bold")
fig.tight_layout()
fig.savefig("plots/raw_grid.png")
print("Saved plots/raw_grid.png")
plt.show()

---

# 4. Extract Priors

Compute prior parameters from the labeled data. Outputs two JSON files:
- `priors_general.json` — purely data-driven means and stds
- `priors_heuristic.json` — same structure with an editable `heuristic_override` field per feature

In [ ]:
import json

PRIOR_FEATURES = [
    "duration_seconds", "mean_plateau_freq_hz", "estimated_cycles",
    "ramp_duration_seconds", "peak_freq_hz", "freq_stability",
    "mean_vibration_intensity", "signal_fraction", "mean_plateau_vibration",
    "mean_autocorr_lag1", "mean_autocorr_lag5",
    "mean_zero_crossing_rate", "mean_peak_to_peak",
]

priors_general = {}
priors_heuristic = {}

for ftype in fixture_types:
    subset = events_df[events_df["fixture_type"] == ftype]
    n = len(subset)

    general_entry = {"n_events": n}
    heuristic_entry = {"n_events": n}

    for feat in PRIOR_FEATURES:
        vals = subset[feat].dropna().values

        if len(vals) == 0:
            mean_val, std_val = 0.0, 1.0
        elif len(vals) == 1:
            mean_val = float(vals[0])
            std_val = max(abs(mean_val) * 0.3, 1.0)  # wide default for n=1
        else:
            mean_val = float(np.mean(vals))
            std_val = float(np.std(vals, ddof=1))
            if std_val < 1e-6:
                std_val = max(abs(mean_val) * 0.1, 0.1)

        general_entry[feat] = {
            "mean": round(mean_val, 4),
            "std": round(std_val, 4),
        }
        heuristic_entry[feat] = {
            "mean": round(mean_val, 4),
            "std": round(std_val, 4),
            "heuristic_override": None,
        }

    priors_general[ftype] = general_entry
    priors_heuristic[ftype] = heuristic_entry

with open("priors_general.json", "w") as f:
    json.dump(priors_general, f, indent=2)
print("Saved priors_general.json")

with open("priors_heuristic.json", "w") as f:
    json.dump(priors_heuristic, f, indent=2)
print("Saved priors_heuristic.json")

# Show what was computed
print(f"\nPrior parameters for {len(fixture_types)} fixture type(s):")
print(json.dumps(priors_general, indent=2))

### How to use `priors_heuristic.json`

The heuristic priors file has the same structure as `priors_general.json` but with an additional `heuristic_override` field per feature. When this field is set (not `null`), the RxInfer model uses it **instead of** the data-derived value.

**When to use overrides:**
- When you have domain knowledge that the data doesn't capture yet (e.g. you know toilets typically run for 30-60 seconds but only have one sample)
- When two fixture types are being confused and you want to sharpen the boundary

**How to set an override:**
Open `priors_heuristic.json` and change `"heuristic_override": null` to a `{"mean": ..., "std": ...}` object:

```json
"mean_plateau_freq_hz": {
    "mean": 0.31,
    "std": 1.0,
    "heuristic_override": {"mean": 0.35, "std": 0.05}
}
```

The model will use `mean=0.35, std=0.05` instead of the data-derived `mean=0.31, std=1.0`.

---

# 5. RxInfer Fixture Classification Model

The Julia code below defines the generative model. Copy `fixture_model.jl` to your Julia environment, or run it in a Julia notebook alongside this one.

The model is a **Gaussian mixture** where:
- The latent variable `fixture_type` selects which component generated the observed features
- Each component has Gaussian priors on each feature, parameterised by the priors JSON files
- Classification = posterior inference over `fixture_type` given observed features

In [ ]:
# The RxInfer model lives in fixture_model.jl (Julia).
# To use it, open a Julia REPL or notebook and run:
#
#   include("Water Fixture Classification/fixture_model.jl")
#   batch_classify("events.csv", "priors_general.json")
#
# Or classify a single event:
#   features = Dict(
#       "duration_seconds" => 44.2,
#       "mean_plateau_freq_hz" => 0.31,
#       "estimated_cycles" => 13.7,
#       "ramp_duration_seconds" => 4.1,
#       "peak_freq_hz" => 0.38,
#       "freq_stability" => 0.12,
#   )
#   predicted, probs, fe = classify(features)

print("Julia model file: Water Fixture Classification/fixture_model.jl")
print("\nPriors files ready:")
for f in ["priors_general.json", "priors_heuristic.json"]:
    if os.path.exists(f):
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ {f} (run Extract Priors cell first)")

---

# Reference

## Adding more labeled data

1. Run the **Labeling Tool** section above to record new events
2. Re-run **Fetch & Stats** — new events appear in `events.csv`
3. Re-run **Extract Priors** — prior parameters update automatically
4. In Julia, reload `fixture_model.jl` — it picks up the new priors

The model automatically incorporates new data. More events = tighter priors = better classification.

## Adding a new fixture type

1. In the labeling tool, type a new label name (add a button or modify the code)
2. Label a few events with the new type
3. Re-run Fetch & Stats — the new type appears in `events.csv`
4. Re-run Extract Priors — a new entry appears in both priors files
5. The Julia model automatically picks up new types from the priors file — no code change needed

## Tuning the heuristic model

Edit `priors_heuristic.json` to set `heuristic_override` on any feature for any fixture type.

| Parameter | What it does |
|-----------|-------------|
| `OBSERVATION_NOISE_SCALE` | Multiplier on feature std. Increase (2.0) = more tolerant. Decrease (0.5) = stricter. |
| `PRIOR_CONFIDENCE` | Weight on the type prior. Increase = harder to override. Decrease = easier to change mind. |
| `FORGETTING_FACTOR` | How fast old data fades in online mode. 1.0 = no forgetting. 0.9 = recent data matters more. |
| `MIN_EVENTS_FOR_RELIABLE_PRIOR` | Warning threshold only. Default 5. |

## Understanding the plots

| Plot | What to look for |
|------|-----------------|
| **raw_signals.png** | The "fingerprint" of each fixture type. Look for distinct shapes — toilets should have a ramp-plateau-cutoff pattern. If two types look identical, classification will be hard. |
| **feature_distributions.png** | Overlap between types. Good separation = histograms don't overlap. With n=1, just shows a single line — need more data. |
| **separation.png** | Quick check: are fixture types in different regions of duration x frequency space? If they cluster together, you need more/different features. |
| **raw_grid.png** | The actual magnetometer waveform. Look for the AC signal from the water meter cog. Flat = no flow detected. Oscillating = flow. |

## Understanding free energy

Free energy measures model surprise. For each `classify()` call:

- **Low (< 5)** — signal matches expectations well. High confidence.
- **Medium (5-15)** — some mismatch but probably correct. Signal had unusual characteristics.
- **High (> 15)** — very surprised. Could mean: unknown fixture type, drifted behaviour, overlapping fixtures, or mislabeled window.

These thresholds are starting points. After collecting data, check your free energy distribution per type and calibrate.

## What to do when accuracy is low

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Low accuracy + low free energy | Features overlap between types | Add heuristic overrides to sharpen boundaries |
| Low accuracy + high free energy | Not enough data | Collect more labeled events |
| One type always wrong | Bad prior initialisation | Check its raw signal plot, adjust override |
| Everything classified as one type | Class imbalance | Collect more events of underrepresented types |

---

# 5b. Test Model

Run the classifier on every labeled event and measure accuracy. This is a sanity check — since we're testing on training data, accuracy should be high. Low accuracy here means the priors are badly separated.

In [ ]:
# ── Classifier function + Test ─────────────────────────────────

import math

def load_priors_py(path="priors_general.json"):
    """Load priors JSON and resolve heuristic overrides."""
    with open(path) as f:
        raw = json.load(f)
    priors = {}
    for ftype, fdata in raw.items():
        priors[ftype] = {"n_events": fdata.get("n_events", 0)}
        for feat in PRIOR_FEATURES:
            if feat not in fdata:
                priors[ftype][feat] = {"mean": 0.0, "std": 1.0}
                continue
            entry = fdata[feat]
            if entry.get("heuristic_override") is not None:
                ov = entry["heuristic_override"]
                priors[ftype][feat] = {"mean": ov["mean"], "std": ov["std"]}
            else:
                priors[ftype][feat] = {"mean": entry["mean"], "std": entry["std"]}
    return priors

def classify_py(features, priors_path="priors_general.json"):
    """Classify features using Gaussian log-likelihood per fixture type."""
    priors = load_priors_py(priors_path)
    fixture_types_list = sorted(priors.keys())

    log_likelihoods = {}
    details = {}
    for ftype in fixture_types_list:
        ll = 0.0
        feat_scores = {}
        for feat in PRIOR_FEATURES:
            p = priors[ftype][feat]
            mu = p["mean"]
            sigma = p["std"]
            if sigma < 1e-8:
                sigma = 1.0
            x = features.get(feat, 0.0)
            if x is None or (isinstance(x, float) and math.isnan(x)):
                x = mu
            x = float(x)
            score = -0.5 * ((x - mu) / sigma) ** 2 - math.log(sigma)
            ll += score
            feat_scores[feat] = {"x": round(x, 4), "mu": round(mu, 4),
                                  "sigma": round(sigma, 4), "score": round(score, 3)}
        log_likelihoods[ftype] = ll
        details[ftype] = feat_scores

    max_ll = max(log_likelihoods.values())
    exp_lls = {}
    for ft, ll in log_likelihoods.items():
        diff = ll - max_ll
        exp_lls[ft] = math.exp(max(-500, min(500, diff)))
    total = sum(exp_lls.values())
    if total == 0:
        total = 1.0
    probs = {ft: exp_lls[ft] / total for ft in fixture_types_list}

    predicted = max(probs, key=probs.get)
    return predicted, probs, details


# ── Test: classify every labeled event ───────────────────────

if not os.path.exists("priors_general.json"):
    print("Run the Extract Priors cell first.")
else:
    test_results = []

    for _, row in events_df.iterrows():
        features = {feat: row[feat] for feat in PRIOR_FEATURES}
        predicted, probs, details = classify_py(features, "priors_general.json")
        conf = probs[predicted]
        correct = predicted == row["fixture_type"]
        test_results.append({
            "signal_id": row["signal_id"],
            "true_label": row["fixture_type"],
            "predicted": predicted,
            "confidence": conf,
            "correct": correct,
        })

    test_df = pd.DataFrame(test_results)
    n_correct = test_df["correct"].sum()
    n_total = len(test_df)
    acc = (n_correct / n_total * 100) if n_total > 0 else 0

    html = ['<div style="font-family:monospace; font-size:14px;">']
    html.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br>')
    html.append('<b>&nbsp;Feature-Based Model — Test on Training Data</b><br>')
    html.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br><br>')

    html.append('<table style="font-family:monospace; font-size:13px; border-collapse:collapse;">')
    html.append('<tr style="border-bottom:1px solid #555;">'
                '<th style="padding:4px 8px;">Signal</th>'
                '<th style="padding:4px 8px;">True</th>'
                '<th style="padding:4px 8px;">Predicted</th>'
                '<th style="padding:4px 8px;">Confidence</th>'
                '<th style="padding:4px 8px;">Correct?</th></tr>')

    for _, r in test_df.iterrows():
        check = '<span style="color:#2ecc71;">yes</span>' if r["correct"] else '<span style="color:#e74c3c;">NO</span>'
        conf_str = f'{r["confidence"]:.1%}'
        html.append(f'<tr>'
                    f'<td style="padding:4px 8px;">#{int(r["signal_id"])}</td>'
                    f'<td style="padding:4px 8px;">{r["true_label"]}</td>'
                    f'<td style="padding:4px 8px; font-weight:bold;">{r["predicted"]}</td>'
                    f'<td style="padding:4px 8px;">{conf_str}</td>'
                    f'<td style="padding:4px 8px;">{check}</td></tr>')

    html.append('</table><br>')
    html.append(f'<b>Accuracy: {n_correct}/{n_total} ({acc:.0f}%)</b><br>')

    if n_total <= 2:
        html.append('<i style="color:#ea0;">Very few events — accuracy is not meaningful yet. '
                    'Label more events and re-run.</i><br>')

    if n_total > 0:
        html.append('<br><b>Per fixture type:</b><br>')
        for ftype in sorted(test_df["true_label"].unique()):
            subset = test_df[test_df["true_label"] == ftype]
            nc = subset["correct"].sum()
            nt = len(subset)
            a = (nc / nt * 100) if nt > 0 else 0
            html.append(f'&nbsp;&nbsp;{ftype}: {nc}/{nt} ({a:.0f}%)<br>')

    if os.path.exists("priors_heuristic.json"):
        heur_correct = 0
        for _, row in events_df.iterrows():
            features = {feat: row[feat] for feat in PRIOR_FEATURES}
            pred_h, _, _ = classify_py(features, "priors_heuristic.json")
            if pred_h == row["fixture_type"]:
                heur_correct += 1
        h_acc = (heur_correct / n_total * 100) if n_total > 0 else 0
        html.append(f'<br><b>Heuristic model: {heur_correct}/{n_total} ({h_acc:.0f}%)</b><br>')

    html.append('</div>')
    display(HTML('\n'.join(html)))

---

# 6. Prediction Tool

Record a fixture event (start/stop), fetch its magnetometer data, extract features, and classify it using the trained priors.
No Julia needed — runs a simple Gaussian log-likelihood classifier in Python using the same priors JSON.

**How to use:**
1. Click **Start** before using a fixture
2. Go use the fixture
3. Come back and click **Stop**
4. The model predicts what it was, with confidence scores

In [ ]:
# ── Prediction Tool ────────────────────────────────────────────

# ── Prediction UI ─────────────────────────────────────────────

pred_start_time = None
pred_end_time = None
pred_timer_running = False

# Widgets
pred_header = widgets.HTML(
    '<div style="font-size:18px; font-weight:bold; font-family:monospace; padding:8px 0;">'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>'
    '&nbsp;Fixture Predictor &nbsp;|&nbsp; sensor 6<br>'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</div>'
)

pred_status = widgets.HTML(value="")
pred_result = widgets.HTML(value="")
pred_details = widgets.HTML(value="")

pred_btn_start = widgets.Button(description="⏺ Start", button_style="danger",
                                 layout=widgets.Layout(width="180px", height="44px"))
pred_btn_stop = widgets.Button(description="⏹ Stop & Predict", button_style="warning",
                                layout=widgets.Layout(width="180px", height="44px"))

pred_priors_toggle = widgets.ToggleButtons(
    options=["priors_general.json", "priors_heuristic.json"],
    value="priors_general.json",
    description="Priors:",
    style={"description_width": "60px"},
)

pred_start_box = widgets.HBox([pred_btn_start, pred_priors_toggle])
pred_stop_box = widgets.HBox([pred_btn_stop])

pred_ui = widgets.VBox([
    pred_header,
    pred_status,
    pred_start_box,
    pred_stop_box,
    pred_result,
    pred_details,
])

def pred_show_idle():
    pred_status.value = '<div style="font-size:14px; font-family:monospace; padding:4px 0;">Press <b>Start</b>, go use a fixture, come back and press <b>Stop & Predict</b>.</div>'
    pred_start_box.layout.display = "flex"
    pred_stop_box.layout.display = "none"

def pred_show_recording():
    pred_start_box.layout.display = "none"
    pred_stop_box.layout.display = "flex"
    pred_result.value = ""
    pred_details.value = ""

def pred_run_timer():
    global pred_timer_running
    while pred_timer_running:
        elapsed = (datetime.now(timezone.utc) - pred_start_time).total_seconds()
        m, s = divmod(int(elapsed), 60)
        pred_status.value = (
            '<div style="font-size:14px; font-family:monospace; padding:4px 0;">'
            f'<span style="color:#e44;">⏺ RECORDING</span> &nbsp; <b>{m:02d}:{s:02d}</b><br>'
            f'Started: {fmt_time(pred_start_time)}<br>'
            'Go use the fixture, then come back and press <b>Stop & Predict</b>.</div>'
        )
        time.sleep(1)

def on_pred_start(_):
    global pred_start_time, pred_timer_running
    pred_start_time = datetime.now(timezone.utc)
    pred_timer_running = True
    pred_show_recording()
    threading.Thread(target=pred_run_timer, daemon=True).start()

def on_pred_stop(_):
    global pred_end_time, pred_timer_running
    pred_timer_running = False
    pred_end_time = datetime.now(timezone.utc)

    dur = (pred_end_time - pred_start_time).total_seconds()
    pred_status.value = (
        '<div style="font-size:14px; font-family:monospace; padding:4px 0;">'
        f'Recorded: {fmt_time(pred_start_time)} → {fmt_time(pred_end_time)} &nbsp; ({fmt_duration(dur)})<br>'
        'Fetching magnetometer data and classifying...</div>'
    )
    pred_start_box.layout.display = "none"
    pred_stop_box.layout.display = "none"

    # Run classification in a thread so UI stays responsive
    threading.Thread(target=run_prediction, daemon=True).start()

def run_prediction():
    try:
        # Fetch mag data for the recorded window
        st = pred_start_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        et = pred_end_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        readings = gql(QUERY_MAG_REPORTS, {
            "sensor_id": SENSOR_ID,
            "start": st,
            "end": et,
        })["mag_report"]

        if len(readings) < 5:
            pred_result.value = (
                '<div style="font-size:15px; font-family:monospace; padding:8px; '
                'color:#e44; border:1px solid #e44; border-radius:6px; margin:8px 0;">'
                f'⚠ Only {len(readings)} magnetometer reading(s) in window. '
                'Need at least 5. Try a longer recording.</div>'
            )
            pred_show_idle()
            return

        # Extract features
        sig_stub = {"id": 0, "value": "?", "start_time": st, "end_time": et}
        features, df, _ = extract_features(sig_stub, readings)

        # Classify
        priors_path = pred_priors_toggle.value
        predicted, probs, details = classify_py(features, priors_path)

        # Build result display
        conf = probs[predicted]
        if conf > 0.7:
            conf_color = "#2ecc71"
            conf_label = "high"
        elif conf > 0.4:
            conf_color = "#f39c12"
            conf_label = "medium"
        else:
            conf_color = "#e74c3c"
            conf_label = "low"

        result_html = (
            '<div style="font-size:18px; font-family:monospace; padding:12px; '
            f'border:2px solid {conf_color}; border-radius:8px; margin:8px 0;">'
            f'<b>Prediction: <span style="color:{conf_color};">{predicted.upper()}</span></b>'
            f' &nbsp; ({conf:.0%} confidence, {conf_label})<br><br>'
        )

        # Probability bars for all types
        for ftype in sorted(probs.keys()):
            p = probs[ftype]
            bar_width = int(p * 200)
            color = FIXTURE_COLORS.get(ftype, "#999")
            result_html += (
                f'<div style="margin:2px 0;">'
                f'<span style="display:inline-block; width:80px;">{ftype}</span>'
                f'<span style="display:inline-block; width:{bar_width}px; height:16px; '
                f'background:{color}; border-radius:3px; margin-right:8px;"></span>'
                f'{p:.1%}</div>'
            )

        result_html += '</div>'
        pred_result.value = result_html

        # Feature detail table
        detail_html = (
            '<details style="font-family:monospace; font-size:12px; margin:8px 0;">'
            '<summary style="cursor:pointer; font-size:13px;"><b>Feature breakdown</b> (click to expand)</summary>'
            '<table style="border-collapse:collapse; margin-top:6px;">'
            '<tr style="border-bottom:1px solid #555;">'
            '<th style="padding:2px 6px; text-align:left;">Feature</th>'
            '<th style="padding:2px 6px; text-align:right;">Observed</th>'
        )
        for ftype in sorted(probs.keys()):
            detail_html += f'<th style="padding:2px 6px; text-align:right;">{ftype} (μ±σ)</th>'
        detail_html += '</tr>'

        for feat in PRIOR_FEATURES:
            x = features.get(feat, 0.0)
            if x is None:
                x = 0.0
            detail_html += f'<tr><td style="padding:2px 6px; color:#6af;">{feat}</td>'
            detail_html += f'<td style="padding:2px 6px; text-align:right;">{x:.4f}</td>'
            for ftype in sorted(probs.keys()):
                d = details[ftype][feat]
                detail_html += (
                    f'<td style="padding:2px 6px; text-align:right;">'
                    f'{d["mu"]:.4f}±{d["sigma"]:.4f}</td>'
                )
            detail_html += '</tr>'

        detail_html += '</table>'

        # Signal quality note
        sf = features.get("signal_fraction", 0)
        mv = features.get("mean_vibration_intensity", 0)
        if sf is not None and sf < 0.3:
            detail_html += (
                '<div style="color:#ea0; margin-top:6px;">⚠ signal_fraction={:.0%} — '
                'most readings were below vibration threshold. '
                'Frequency features may be unreliable.</div>'.format(sf)
            )
        if mv is not None and mv < MIN_VIBRATION:
            detail_html += (
                '<div style="color:#ea0; margin-top:4px;">⚠ mean vibration={:.1f} < {} — '
                'event may be entirely noise.</div>'.format(mv, MIN_VIBRATION)
            )

        detail_html += '</details>'
        pred_details.value = detail_html

    except Exception as e:
        pred_result.value = (
            '<div style="font-size:14px; font-family:monospace; padding:8px; '
            f'color:#e44; border:1px solid #e44; border-radius:6px; margin:8px 0;">'
            f'Error: {e}</div>'
        )

    pred_show_idle()

pred_btn_start.on_click(on_pred_start)
pred_btn_stop.on_click(on_pred_stop)

pred_show_idle()
display(pred_ui)